# 04 – Tiền xử lý & đặc trưng văn bản; tinh chỉnh Naive Bayes, Logistic Regression

**Người thực hiện:** loc (Huỳnh Tấn Lộc)  
**Mục tiêu:** Trả lời: tiền xử lý / n-gram / min_df ảnh hưởng thế nào? Chọn cấu hình NB, LR tốt nhất.

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> loc`.

**Các bước:**
1. Kiểm chứng lại các con số dữ liệu dùng trong báo cáo (tự giải thích ý nghĩa).
2. Thử word_ngram (1,1), (1,2), (1,3) với NB.
3. Thử min_df 1, 2, 5.
4. Quét alpha của NB (ít nhất 5 giá trị).
5. Quét C của Logistic Regression (ít nhất 5 giá trị), có/không class_weight='balanced'.
6. Vẽ đường F1 theo tham số (lưu bằng save_fig) và chọn cấu hình tốt nhất.

In [1]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


## 1. Kiểm chứng số liệu dữ liệu
Mỗi con số trong báo cáo phải chỉ được ra ô code sinh ra nó. Chạy ô dưới, rồi **tự viết** 1–2 câu giải thích ý nghĩa từng số.

In [2]:
df = load_data()
print('Số mẫu:', len(df))
print('Số merchant:', df.merchant_id.nunique())
print('Số cụm thực thể:', df.cluster_id.nunique())
print('Giá trị khuyết:', df.isna().sum().sum())
df.category_label.value_counts()

Số mẫu: 35311
Số merchant: 306
Số cụm thực thể: 13233
Giá trị khuyết: 0


category_label
Fridge Freezers     5501
Mobile Phones       4081
Washing Machines    4044
CPUs                3862
Fridges             3584
TVs                 3564
Dishwashers         3424
Digital Cameras     2697
Microwaves          2342
Freezers            2212
Name: count, dtype: int64

**Giải thích (tự viết):**

- **35.311 mẫu**: mỗi dòng là một tin rao bán sản phẩm trên trang so sánh giá PriceRunner, gồm tên sản phẩm (title), cửa hàng đăng bán và danh mục. Mô hình chỉ dùng title để đoán danh mục.
- **306 merchant**: dữ liệu được gom từ 306 cửa hàng khác nhau. Mỗi cửa hàng đặt tên sản phẩm theo cách riêng (thứ tự từ, viết tắt, mã model), nên cùng một loại sản phẩm có nhiều cách viết khác nhau, khiến bài toán khó hơn.
- **13.233 cụm thực thể**: số sản phẩm thật sự khác nhau. Trung bình mỗi sản phẩm xuất hiện khoảng 2,7 lần, do nhiều cửa hàng cùng bán một sản phẩm. Đây là nhãn thật (ground truth) dùng để đánh giá phần phân cụm.
- **0 giá trị khuyết**: dữ liệu đầy đủ, không cần xử lý thiếu.
- **Phân bố lớp**: lớp nhiều nhất là **Fridge Freezers (5.501 mẫu, khoảng 15,6%)**, lớp ít nhất là **Freezers (2.212 mẫu, khoảng 6,3%)**, chênh nhau khoảng 2,5 lần. Dữ liệu lệch lớp ở mức vừa phải, vì vậy nhóm dùng **Macro-F1** (mỗi lớp có trọng số như nhau) thay vì chỉ dùng Accuracy. Ba lớp tủ lạnh (Fridges, Freezers, Fridge Freezers) có tên sản phẩm gần giống nhau, nên đây là chỗ mô hình dễ nhầm nhất, khớp với kết quả thí nghiệm NB của mình.

In [3]:
# Ví dụ làm sạch tiêu đề: so sánh trước / sau
for t in df.title.sample(8, random_state=0):
    print(f'{t!r:70s} -> {clean_title(t)!r}')

'kodak az651 65 multiplier x'                                          -> 'kodak az651 65 multiplier x'
'beko csg1675w 175x60cm 273l freestanding fridge freezer white'        -> 'beko csg1675w 175x60cm 273l freestanding fridge freezer white'
'datalogic powerscan pbt9500 handheld 1d/2d photo diode black yellow'  -> 'datalogic powerscan pbt9500 handheld 1d 2d photo diode black yellow'
'amd 7th gen amd athlon x4 950 processor'                              -> 'amd 7th gen amd athlon x4 950 processor'
'siemens geschirrsp ler 45 cm speedmatic45 umbra unterbau sr36t498eu'  -> 'siemens geschirrsp ler 45cm speedmatic45 umbra unterbau sr36t498eu'
'whirlpool arg18083a 178cm integrated in column larder fridge'         -> 'whirlpool arg18083a 178cm integrated in column larder fridge'
'hoover hvbfp6182w/hvbfp6182w'                                         -> 'hoover hvbfp6182w hvbfp6182w'
'samsung wf80f5e0w2w washing machine'                                  -> 'samsung wf80f5e0w2w washing machine'


**Nhận xét: hàm clean_title giữ lại gì, bỏ đi gì, tại sao? (tự viết)** 

- **Giữ lại**: tên hãng (beko, whirlpool, samsung...), mã model (csg1675w, wf80f5e0w2w...), con số và đơn vị (273l, 175x60cm, 178cm), kể cả từ không phải tiếng Anh (geschirrsp ler, unterbau). Đây là những thông tin phân biệt danh mục rất mạnh: dung tích "273l" gợi ý tủ lạnh, "45cm" là kích thước điển hình của máy rửa bát, mã model thường có tiền tố riêng theo từng loại sản phẩm.
- **Thay đổi / bỏ đi**:
  - Dấu "/" được thay bằng dấu cách: "1d/2d" thành "1d 2d", "hvbfp6182w/hvbfp6182w" thành 2 từ riêng. Nhờ vậy TF-IDF nhận ra từng mã như một từ bình thường, không bị dính thành một từ lạ chỉ xuất hiện một lần.
  - Số và đơn vị bị tách được ghép lại: "45 cm" thành "45cm". Mỗi cửa hàng viết một kiểu (có hoặc không có dấu cách), ghép lại giúp cùng một thông tin luôn thành cùng một từ.
- **Tại sao làm nhẹ tay như vậy**: dữ liệu đã viết thường sẵn, và tên sản phẩm rất ngắn nên mỗi từ đều có giá trị. Hàm chỉ chuẩn hóa cách viết, không xóa số hay mã model như cách làm sạch văn bản thông thường. Nếu bỏ số và mã model thì mô hình mất đi đặc trưng quan trọng nhất.

## 2. Thí nghiệm n-gram (Naive Bayes)
Đổi `word_ngram` rồi chạy lại ô. Mỗi lần chạy = 1 dòng nhật ký.

In [ ]:
res, _ = run(member='loc', model='nb', features='word', params={'alpha': 0.1},
             word_ngram=(1, 1), exp_name='ngram_11', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 3. Thí nghiệm min_df

In [ ]:
res, _ = run(member='loc', model='nb', features='word', params={'alpha': 0.1},
             min_df=1, exp_name='mindf_1', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. Quét alpha (Naive Bayes)
Chạy từng giá trị (hoặc vòng lặp nhỏ) – ghi lại nhận xét sau mỗi đợt.

In [ ]:
for a in [1.0, 0.5]:   # lần sau thử: 0.1, 0.05, 0.01 ...
    run(member='loc', model='nb', features='word', params={'alpha': a}, exp_name=f'alpha_{a}', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 5. Quét C (Logistic Regression)

In [ ]:
run(member='loc', model='lr', features='word', params={'C': 1.0}, exp_name='lr_C1', show_report=False)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/loc/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 6. Tổng hợp & vẽ
Đọc lại nhật ký của mình, vẽ F1 theo tham số.

In [ ]:
log = pd.read_csv('../experiments/loc/experiment_log.csv')
nb = log[log.model == 'nb'].copy()
nb['alpha'] = nb.params.map(lambda s: json.loads(s).get('alpha'))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(nb.alpha, nb.test_f1_macro, 'o')
ax.set(xscale='log', xlabel='alpha', ylabel='Macro-F1 (Test)', title='NB: Macro-F1 theo alpha')
save_fig(fig, 'loc_nb_alpha', notebook='04_Loc_TienXuLy_NB_LR.ipynb', member='loc', cell='Mục 6',
         y_nghia='(tự viết ý nghĩa hình)')

## 7. Kết luận của Lộc (tự viết)
- Cấu hình chọn: …
- Vì sao: …
- Hạn chế: …